<a id="ip-overview"></a>

$$
\text{整数规划 (Integer Programming, IP): 部分或全部决策变量必须取整数的线性规划} \\
\max c^T x \quad \text{s.t.} \quad A x \leq b, \ x \geq 0, \ x_j \in \mathbb{Z} \\
\text{全部为整数: 纯整数规划; 部分为整数: 混合整数规划 (Mixed Integer Linear Programming, MILP)} \\
\text{变量只能取 0 或 1: 0-1 规划 (选或不选, 做或不做)} \\ \quad \\
\text{去掉整数约束得到的线性规划叫松弛问题 (LP Relaxation)} \\
\text{松弛问题的可行域包含了所有整数可行解, 所以 } z^*_{LP} \geq z^*_{IP} \text{ (松弛问题给出上界)} \\ \quad \\
\text{为什么不能直接把 LP 的解四舍五入? 例:} \\
\max z = 21 x_1 + 11 x_2 \quad \text{s.t.} \quad 7 x_1 + 4 x_2 \leq 13, \ x_1, x_2 \geq 0 \text{ 且为整数} \\
\text{LP 松弛: 单位资源的价值 } \frac{21}{7} = 3 > \frac{11}{4} = 2.75 \text{, 全部给 } x_1 \implies x_1 = \frac{13}{7} \approx 1.857, \ x_2 = 0, \ z = 39 \\
\text{四舍五入 (2, 0): } 7 \times 2 = 14 > 13 \text{, 不可行} \\
\text{向下取整 (1, 0): z = 21} \\
\text{真正的最优解 (0, 3): } 4 \times 3 = 12 \leq 13 \text{, z = 33, 离 LP 的解很远}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import permutations, product
from scipy.optimize import linprog, milp, LinearConstraint, Bounds, linear_sum_assignment

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

c_ex = np.array([21.0, 11.0])
A_ex = np.array([[7.0, 4.0]])
b_ex = np.array([13.0])

lp = linprog(-c_ex, A_ub=A_ex, b_ub=b_ex, method='highs')
print(f'LP 松弛: x = {lp.x}, z = {-lp.fun:.4f}')

# 枚举所有整数点
points = np.array([(i, j) for i in range(3) for j in range(5) if 7 * i + 4 * j <= 13])
values = points @ c_ex
best = points[np.argmax(values)]
print(f'所有可行整数点: {points.tolist()}')
print(f'枚举的最优整数解 = {best}, z = {values.max():.0f}')
print(f'四舍五入 {np.round(lp.x)}: 7x1 + 4x2 = {A_ex @ np.round(lp.x)} > 13, 不可行; 向下取整 {np.floor(lp.x)}: z = {c_ex @ np.floor(lp.x):.0f}')

x_line = np.linspace(0, 2.2, 50)
plt.figure(figsize=(4, 3))
plt.fill_between(x_line, 0, np.clip((13 - 7 * x_line) / 4, 0, None), alpha=0.3, label='LP feasible region')
grid = np.array([(i, j) for i in range(3) for j in range(5)])
plt.plot(grid[:, 0], grid[:, 1], 'k.', markersize=4)
plt.plot(points[:, 0], points[:, 1], 'bo', label='feasible integer points')
plt.plot(*lp.x, 'r*', markersize=12, label='LP optimum')
plt.plot(*best, 'gs', markersize=9, label='IP optimum')
plt.xlabel('x1')
plt.ylabel('x2')
plot_nicely()
plt.show()

<a id="branch-and-bound"></a>

$$
\text{分支定界法 (Branch and Bound)} \\ \quad \\
\text{1. 定界 (Bound): 解当前子问题的 LP 松弛, 得到这个子问题能达到的上界 } \bar{z} \\
\text{2. 分支 (Branch): 若 LP 解中某个 } x_j = v \text{ 不是整数, 拆成两个子问题} \\
\quad x_j \leq \lfloor v \rfloor \quad \text{或} \quad x_j \geq \lceil v \rceil \\
\quad \text{两个子问题合起来包含了原问题所有的整数解 (} \lfloor v \rfloor \text{ 和 } \lceil v \rceil \text{ 之间没有整数), 但去掉了当前的分数解} \\
\text{3. 剪枝 (Prune): 记录目前找到的最好整数解 (incumbent) 的值 } z_{best} \\
\quad \text{(a) LP 无可行解: 剪掉} \\
\quad \text{(b) } \bar{z} \leq z_{best} \text{: 这个子问题里最好也不会超过当前最优, 剪掉 (定界的作用)} \\
\quad \text{(c) LP 解恰好是整数: 它就是这个子问题的最优解, 更新 } z_{best} \text{, 不用再分} \\ \quad \\
\text{正确性: 被剪掉的子问题里不可能有比 } z_{best} \text{ 更好的解, 而没被剪掉的都继续搜索, 所以结束时 } z_{best} \text{ 就是全局最优}
$$

In [ ]:
def branch_and_bound(c, A_ub, b_ub, bounds, verbose=True):
    # max c^T x, A_ub x <= b_ub, 所有变量为整数, bounds = [(lo, hi), ...]
    best = {'z': -np.inf, 'x': None, 'nodes': 0}

    def solve(bounds, depth):
        best['nodes'] += 1
        res = linprog(-c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
        pad = '    ' * depth
        if res.status != 0:
            if verbose: print(f'{pad}{bounds}: LP 无可行解, 剪枝')
            return
        z, x = -res.fun, res.x
        if z <= best['z'] + 1e-9:
            if verbose: print(f'{pad}{bounds}: 上界 {z:.4f} <= 当前最优 {best["z"]:.4f}, 剪枝')
            return
        frac = [j for j in range(len(x)) if abs(x[j] - np.round(x[j])) > 1e-6]
        if not frac:
            best['z'], best['x'] = z, np.round(x)
            if verbose: print(f'{pad}{bounds}: 整数解 x = {np.round(x)}, z = {z:.4f}, 更新当前最优')
            return
        j = frac[0]
        if verbose: print(f'{pad}{bounds}: LP x = {np.round(x, 4)}, z = {z:.4f}, 对 x{j + 1} = {x[j]:.4f} 分支')
        lo, hi = bounds[j]
        left = list(bounds); left[j] = (lo, int(np.floor(x[j])))
        right = list(bounds); right[j] = (int(np.ceil(x[j])), hi)
        solve(left, depth + 1)
        solve(right, depth + 1)

    solve(list(bounds), 0)
    return best['x'], best['z'], best['nodes']

x_bb, z_bb, nodes = branch_and_bound(c_ex, A_ex, b_ex, [(0, None), (0, None)])
print(f'\n分支定界: x = {x_bb}, z = {z_bb:.0f}, 共解了 {nodes} 个 LP')

m = milp(-c_ex, constraints=LinearConstraint(A_ex, -np.inf, b_ex), integrality=np.ones(2), bounds=Bounds(0, np.inf))
print(f'scipy milp: x = {m.x}, z = {-m.fun:.0f}, 一致? {np.isclose(z_bb, -m.fun)}')

<a id="knapsack"></a>

$$
\text{0-1 背包问题 (0-1 Knapsack Problem)} \\
\max \sum_{i=1}^n v_i x_i \quad \text{s.t.} \quad \sum_{i=1}^n w_i x_i \leq W, \quad x_i \in \{0, 1\} \\
v_i \text{: 价值, } w_i \text{: 重量, W: 背包容量} \\ \quad \\
\text{LP 松弛 } (0 \leq x_i \leq 1) \text{ 的解可以直接写出来 (Dantzig):} \\
\text{按性价比 } \frac{v_i}{w_i} \text{ 从大到小排序, 依次装满, 最后一个装不下的物品装一部分} \\
\text{理由 (交换论证): 若性价比高的 i 没装满而性价比低的 k 装了一部分, 把 k 的重量 } \delta \text{ 换给 i} \\
\text{价值变化 } = \delta \left( \frac{v_i}{w_i} - \frac{v_k}{w_k} \right) > 0 \text{, 所以最优解一定先装性价比高的} \\ \quad \\
\text{但 0-1 版本的贪心 (按性价比装, 装不下就跳过) 不一定最优: 可能留下空间浪费} \\
\text{暴力枚举需要检查 } 2^n \text{ 种组合; 分支定界用 LP 的上界剪掉大量组合}
$$

In [ ]:
values_k = np.array([24, 13, 23, 15, 16, 11, 8, 26, 5, 19, 17, 9], dtype=float)
weights_k = np.array([12, 7, 11, 8, 9, 6, 5, 14, 3, 10, 9, 4], dtype=float)
capacity = 42.0
n_items = len(values_k)

# 1. 暴力枚举 2^n 种组合
all_x = np.array(list(product([0, 1], repeat=n_items)), dtype=float)
feasible = all_x @ weights_k <= capacity
brute_values = np.where(feasible, all_x @ values_k, -np.inf)
x_brute = all_x[np.argmax(brute_values)]
print(f'暴力枚举 {2**n_items} 种组合: 最优价值 = {brute_values.max():.0f}, 选中 = {np.flatnonzero(x_brute).tolist()}')

# 2. 贪心 (按性价比)
order = np.argsort(-values_k / weights_k)
x_greedy, room = np.zeros(n_items), capacity
for i in order:
    if weights_k[i] <= room:
        x_greedy[i], room = 1, room - weights_k[i]
print(f'贪心: 价值 = {x_greedy @ values_k:.0f}, 剩余容量 = {room:.0f} (不是最优, 剩下的空间装不下任何还没装的物品)')

# 3. LP 松弛 = Dantzig 的分数贪心
lp_k = linprog(-values_k, A_ub=weights_k[None, :], b_ub=[capacity], bounds=[(0, 1)] * n_items, method='highs')
x_frac, room = np.zeros(n_items), capacity
for i in order:
    take = min(1.0, room / weights_k[i])
    x_frac[i], room = take, room - take * weights_k[i]
    if room <= 0: break
print(f'LP 松弛上界 = {-lp_k.fun:.4f}, 分数贪心 = {x_frac @ values_k:.4f}')

# 4. 分支定界
x_bb_k, z_bb_k, nodes_k = branch_and_bound(values_k, weights_k[None, :], [capacity], [(0, 1)] * n_items, verbose=False)
print(f'分支定界: 价值 = {z_bb_k:.0f}, 只解了 {nodes_k} 个 LP (暴力枚举是 {2**n_items} 种)')

# 5. scipy milp
m_k = milp(-values_k, constraints=LinearConstraint(weights_k[None, :], -np.inf, capacity),
           integrality=np.ones(n_items), bounds=Bounds(0, 1))
print(f'scipy milp: 价值 = {-m_k.fun:.0f}, 选中 = {np.flatnonzero(np.round(m_k.x)).tolist()}')
print(f'三种精确方法一致? {np.isclose(brute_values.max(), z_bb_k) and np.isclose(z_bb_k, -m_k.fun)}')

<a id="assignment"></a>

$$
\text{指派问题 (Assignment Problem): n 个人做 n 项工作, 每人一项, 每项一人, 总成本最小} \\
\min \sum_{i=1}^n \sum_{j=1}^n c_{ij} x_{ij} \\
\text{s.t.} \quad \sum_{j=1}^n x_{ij} = 1 \ (\text{每个人只做一项}), \quad \sum_{i=1}^n x_{ij} = 1 \ (\text{每项只由一个人做}), \quad x_{ij} \in \{0, 1\} \\ \quad \\
\text{x 是一个置换矩阵 (Permutation Matrix), 所以暴力法要枚举 n! 种排列} \\
\text{(n = 10 已经有 3628800 种, 实际用匈牙利算法 Hungarian Algorithm, 复杂度 } O(n^3) \text{)} \\ \quad \\
\text{一个特殊性质: 指派问题的 LP 松弛 } (0 \leq x_{ij} \leq 1) \text{ 的顶点全部是整数解} \\
\text{原因: 约束矩阵是全幺模矩阵 (Totally Unimodular), 所有子方阵的行列式 } \in \{0, \pm 1\} \\
\text{顶点 = } B^{-1} b \text{, 由克莱姆法则 (Cramer's Rule) 分母是 } \det B = \pm 1 \text{, b 是整数} \implies \text{顶点是整数} \\
\text{所以指派问题直接用线性规划就能得到整数解}
$$

数学基础: [行列式](../math_basics/linear_algebra.ipynb#determinant) · [矩阵的逆](../math_basics/linear_algebra.ipynb#inverse)

In [ ]:
rng = np.random.default_rng(2)
n_a = 7
cost_a = rng.integers(10, 60, size=(n_a, n_a)).astype(float)
print(f'成本矩阵 = \n{cost_a}')

# 1. 暴力枚举 n! 种排列
best_cost, best_perm = np.inf, None
for perm in permutations(range(n_a)):
    total = cost_a[np.arange(n_a), perm].sum()
    if total < best_cost:
        best_cost, best_perm = total, perm
print(f'暴力枚举 {n_a}! = {len(list(permutations(range(n_a))))} 种: 最小成本 = {best_cost:.0f}, 分配 = {list(best_perm)}')

# 2. 匈牙利算法
rows, cols = linear_sum_assignment(cost_a)
print(f'linear_sum_assignment: 最小成本 = {cost_a[rows, cols].sum():.0f}, 分配 = {cols.tolist()}')

# 3. LP 松弛 (不加整数约束), 解自动是 0/1
A_rows = np.kron(np.eye(n_a), np.ones(n_a))      # 每行 x_ij 的和
A_cols = np.kron(np.ones(n_a), np.eye(n_a))      # 每列 x_ij 的和
lp_a = linprog(cost_a.ravel(), A_eq=np.vstack([A_rows, A_cols]), b_eq=np.ones(2 * n_a), bounds=(0, 1), method='highs')
X_lp = lp_a.x.reshape(n_a, n_a)
print(f'LP 松弛: 最小成本 = {lp_a.fun:.0f}, 解全部是 0 或 1? {np.allclose(X_lp, np.round(X_lp))}')
print(f'x 矩阵 (置换矩阵) = \n{np.round(X_lp).astype(int)}')

# 全幺模的随机抽查: 随机子方阵的行列式
A_full = np.vstack([A_rows, A_cols])
dets = set()
for _ in range(3000):
    k = rng.integers(1, 6)
    r = rng.choice(A_full.shape[0], k, replace=False)
    cidx = rng.choice(A_full.shape[1], k, replace=False)
    dets.add(int(np.round(np.linalg.det(A_full[np.ix_(r, cidx)]))))
print(f'3000 个随机子方阵的行列式取值 = {sorted(dets)}')